In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
RANDOM_STATE = 42

train = pd.read_csv('../data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('../data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('../data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


### Data overview

In [2]:
train.info()
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 11091 entries, 0 to 11090
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   cookie_id          11091 non-null  str           
 1   cookie_created_at  11091 non-null  datetime64[us]
 2   window_start_ts    11091 non-null  datetime64[us]
 3   window_end_ts      11091 non-null  datetime64[us]
 4   target             11091 non-null  int64         
dtypes: datetime64[us](3), int64(1), str(1)
memory usage: 433.4 KB
<class 'pandas.DataFrame'>
RangeIndex: 328905 entries, 0 to 328904
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   cookie_id      328905 non-null  str           
 1   event_ts       328905 non-null  datetime64[us]
 2   eid            328905 non-null  int64         
 3   event_name     328905 non-null  str           
 4   platform       328905 non-null  st

In [3]:
events['event_name'].value_counts()


event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64

### События внутри заданного окна

In [4]:
columns_to_keep = ['cookie_id', 'window_start_ts','window_end_ts' ]
windows = pd.concat([train[columns_to_keep], test[columns_to_keep]],ignore_index=True)

In [5]:
events_with_windows = pd.merge(windows,events,on='cookie_id',how='right')

In [6]:
mask = (events_with_windows['window_start_ts'] <= events_with_windows['event_ts']) & (events_with_windows['event_ts'] < events_with_windows['window_end_ts'])
events_filtered = events_with_windows[mask]

In [7]:
events.shape, events_filtered.shape

((328905, 14), (288126, 16))

## Признаки

В качестве начальных признаков берем:
- общее число событий
- число уникальных объявлений
- количество событий каждого типа
- число уникальных категорий
- число уникальных локаций

In [8]:
n_unique_items = events_filtered.groupby('cookie_id')['item_id'].nunique()
n_events = events_filtered.groupby('cookie_id').size()
features = pd.concat([n_events, n_unique_items], axis=1)
features.columns = ['n_events', 'n_unique_items']

In [9]:
event_counts = events_filtered.groupby(['cookie_id', 'event_name']).size()
event_counts = event_counts.unstack(fill_value=0)
event_counts = event_counts.add_prefix('n_')

In [10]:
features = pd.concat([features, event_counts], axis=1)

In [11]:
n_unique_location = events_filtered.groupby('cookie_id')['item_location'].nunique()
n_unique_category = events_filtered.groupby('cookie_id')['item_category'].nunique()

In [12]:
features['item_location'] = n_unique_location
features['item_category'] = n_unique_category

In [13]:
features['favourite_share'] = features['n_favorite_add'] / features['n_events']
features

,n_events,n_unique_items,n_contact_chat_open,n_contact_message_sent,n_contact_phone_show,n_favorite_add,n_item_view,n_login,n_photo_swipe,n_search_results_view,n_seller_page_view,item_location,item_category,favourite_share
cookie_id,,,,,,,,,,,,,,
ck_00013fdfa0bd37dd,15,9,2,0,0,1,4,2,3,3,0,4,3,0.066667
ck_000c95f1408dcb00,16,9,0,0,0,0,11,0,1,3,1,8,2,0.000000
ck_000e8c52636e3bec,34,20,0,0,1,4,11,1,5,9,3,12,4,0.117647
ck_0010e31baa4a1fb7,16,8,0,0,0,2,7,1,1,4,1,4,3,0.125000
ck_0010ec3874fb5378,74,31,0,0,0,10,25,1,14,20,4,19,2,0.135135
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ck_ffee36a37e2532f5,81,38,1,2,6,1,27,0,15,24,5,23,4,0.012346
ck_fff88a7920fd0fb6,11,6,0,0,0,0,8,0,1,2,0,3,1,0.000000
ck_fff9fdc2adf5100b,37,22,0,1,7,0,19,2,4,2,2,11,1,0.000000


In [14]:
new = pd.concat([train, test], ignore_index=True).set_index('cookie_id')
new

,cookie_created_at,window_start_ts,window_end_ts,target
cookie_id,,,,
ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0.0
ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0.0
ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0.0
ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0.0
ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0.0
...,...,...,...,...
ck_8b772caed8f1541c,2026-03-06 06:17:32,2026-04-26,2026-04-27,NaN
ck_5403c76fe68a08ba,2025-09-16 01:17:03,2026-04-26,2026-04-27,NaN
ck_3e6833574edf1736,2026-02-06 19:45:02,2026-04-26,2026-04-27,NaN


In [15]:
features['cookie_age_id'] = (new['window_end_ts'] - new['cookie_created_at']).dt.total_seconds() / 86400
features

,n_events,n_unique_items,n_contact_chat_open,n_contact_message_sent,n_contact_phone_show,n_favorite_add,n_item_view,n_login,n_photo_swipe,n_search_results_view,n_seller_page_view,item_location,item_category,favourite_share,cookie_age_id
cookie_id,,,,,,,,,,,,,,,
ck_00013fdfa0bd37dd,15,9,2,0,0,1,4,2,3,3,0,4,3,0.066667,197.215428
ck_000c95f1408dcb00,16,9,0,0,0,0,11,0,1,3,1,8,2,0.000000,233.960440
ck_000e8c52636e3bec,34,20,0,0,1,4,11,1,5,9,3,12,4,0.117647,26.480150
ck_0010e31baa4a1fb7,16,8,0,0,0,2,7,1,1,4,1,4,3,0.125000,83.715069
ck_0010ec3874fb5378,74,31,0,0,0,10,25,1,14,20,4,19,2,0.135135,103.395208
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ck_ffee36a37e2532f5,81,38,1,2,6,1,27,0,15,24,5,23,4,0.012346,112.370810
ck_fff88a7920fd0fb6,11,6,0,0,0,0,8,0,1,2,0,3,1,0.000000,59.496227
ck_fff9fdc2adf5100b,37,22,0,1,7,0,19,2,4,2,2,11,1,0.000000,1.550058


In [16]:
features.columns.tolist()

['n_events',
 'n_unique_items',
 'n_contact_chat_open',
 'n_contact_message_sent',
 'n_contact_phone_show',
 'n_favorite_add',
 'n_item_view',
 'n_login',
 'n_photo_swipe',
 'n_search_results_view',
 'n_seller_page_view',
 'item_location',
 'item_category',
 'favourite_share',
 'cookie_age_id']

In [17]:
base_cols = features.columns.tolist()

### Временные признаки

In [18]:
events_sorted = events_filtered.sort_values(['cookie_id', 'event_ts'])

In [19]:
events_sorted['seconds_since_prev_event'] = (events_sorted.groupby('cookie_id')['event_ts'].diff().dt.total_seconds())

In [20]:
events_sorted[['cookie_id', 'event_ts', 'seconds_since_prev_event']].head(10)

,cookie_id,event_ts,seconds_since_prev_event
67286,ck_00013fdfa0bd37dd,2026-04-24 18:41:16,NaN
72380,ck_00013fdfa0bd37dd,2026-04-24 19:16:53,2137.0
189910,ck_00013fdfa0bd37dd,2026-04-24 19:16:54,1.0
104718,ck_00013fdfa0bd37dd,2026-04-24 19:17:05,11.0
237561,ck_00013fdfa0bd37dd,2026-04-24 20:05:05,2880.0
144174,ck_00013fdfa0bd37dd,2026-04-24 20:31:33,1588.0
324747,ck_00013fdfa0bd37dd,2026-04-24 20:34:06,153.0
249330,ck_00013fdfa0bd37dd,2026-04-24 20:36:30,144.0
278367,ck_00013fdfa0bd37dd,2026-04-24 20:36:32,2.0
28960,ck_00013fdfa0bd37dd,2026-04-24 20:41:33,301.0


In [21]:
features['gap_median'] = (events_sorted.groupby('cookie_id')['seconds_since_prev_event'].median())

In [22]:
features['gap_median'].isna().sum()

np.int64(351)

In [23]:
features.loc[features['gap_median'].isna(), 'n_events'].value_counts()

n_events
1    351
Name: count, dtype: int64

In [24]:
features['gap_missing'] = features['gap_median'].isna().astype(int)
features['gap_median'] = features['gap_median'].fillna(0)

In [25]:
features['gap_std'] = (events_sorted.groupby('cookie_id')['seconds_since_prev_event'].std())

In [26]:
gap_cols = base_cols + ['gap_median', 'gap_missing']
std_cols = gap_cols + ['gap_std']

## Validation

Тестовая выборка расположена во времени после обучающей, поэтому случайное разбиение не соответствует реальному сценарию

Для валидации используются последние три дня обучающей выборки:
- train: 6–16 апреля 2026
- validation: 17–19 апреля 2026

In [27]:
VALID_START = pd.Timestamp('2026-04-17')
train_mask = train['window_start_ts'] < VALID_START
valid_mask = train['window_start_ts'] >= VALID_START

train_ids = train.loc[train_mask, 'cookie_id']
valid_ids = train.loc[valid_mask, 'cookie_id']

In [28]:
X_train = features.loc[train_ids]
X_valid= features.loc[valid_ids]

y_train = train.loc[train_mask, 'target']
y_valid = train.loc[valid_mask, 'target']

In [29]:
X_train.shape, X_valid.shape

((9140, 18), (1951, 18))

In [30]:
features.columns.tolist()

['n_events',
 'n_unique_items',
 'n_contact_chat_open',
 'n_contact_message_sent',
 'n_contact_phone_show',
 'n_favorite_add',
 'n_item_view',
 'n_login',
 'n_photo_swipe',
 'n_search_results_view',
 'n_seller_page_view',
 'item_location',
 'item_category',
 'favourite_share',
 'cookie_age_id',
 'gap_median',
 'gap_missing',
 'gap_std']

## Обучение и оценка

В качестве базовой модели используется логистическая регрессия

Качество оценивается официальной метрикой Precision при Recall ≥ 70%

In [31]:
from sklearn.linear_model import LogisticRegression

In [32]:
import sys

sys.path.append('..')

from metric import precision_at_recall

In [33]:
X_train.isna().sum()[X_train.isna().any()]

gap_std    536
dtype: int64

In [34]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=RANDOM_STATE, max_iter=2000)
)

model.fit(X_train[gap_cols], y_train)
pred = model.predict_proba(X_valid[gap_cols])[:, 1]
precision_at_recall(y_valid, pred)

0.23689727463312368

In [35]:
model.fit(X_train[base_cols], y_train)
pred = model.predict_proba(X_valid[base_cols])[:, 1]
precision_at_recall(y_valid, pred)

0.24669603524229075

## Градиентный бустинг
Сравниваем с логистической регрессией на тех же признаках

In [36]:
from sklearn.ensemble import HistGradientBoostingClassifier

X_train_boost = X_train[base_cols]
X_valid_boost = X_valid[base_cols]

boost_model = HistGradientBoostingClassifier(
    random_state=RANDOM_STATE
)

boost_model.fit(X_train_boost, y_train)
pred_boost = boost_model.predict_proba(X_valid_boost)[:, 1]

precision_at_recall(y_valid, pred_boost)

0.24835164835164836

In [37]:
boost_with_gaps = HistGradientBoostingClassifier(
    random_state=RANDOM_STATE
)
boost_with_gaps.fit(X_train[gap_cols], y_train)
pred_boost_gaps = boost_with_gaps.predict_proba(X_valid[gap_cols])[:, 1]
precision_at_recall(y_valid, pred_boost_gaps)

0.35

In [38]:
boost_with_std = HistGradientBoostingClassifier(
    random_state=RANDOM_STATE
)
boost_with_std.fit(X_train[std_cols], y_train)
pred_std = boost_with_std.predict_proba(X_valid[std_cols])[:, 1]
precision_at_recall(y_valid, pred_std)

0.3783783783783784

## Результаты экспериментов

| Модель | Без интервалов | + gap_median и gap_missing | + gap_std |
|---|---:|---:|---:|
| Logistic Regression + StandardScaler | 0.2467 | 0.2369 | - |
| Градиентный бустинг | 0.24835 | 0.35 | 0.3784 |

Медиана интервалов и индикатор её отсутствия улучшили качество
бустинга, но ухудшили результат логистической регрессии.
Добавление стандартного отклонения интервалов повысило метрику
бустинга с 0.35 до 0.3784 на выбранной временной валидации.